# LFM Semantic Segmentation Example Workflow
This notebook is an example workflow of doing semantic segmentation on visible light, UV, and static bands of Lunar data. 

## Purpose of this notebook
This notebook is designed to be used as an example semantic segmentation workflow ("crater" vs "non crater" model prediction). A pretrained DinoV3 model is loaded from disk, and we build our own crater detection model on top of this. The model loads data from the lfm project space, then runs several epochs of training on this data, and finally visualizes the model performance on the validation dataset. If you would like to control some of the model parameters, see the "Config" section below.  

**Note**: currently, the training dataset used in this notebooks is comprised of 12-band input data (5 VIS bands, 2 UV bands, 5 Kaguya static bands). If you wish to train your own model with a different input dataset (i.e. different static band configuration), that will be added soon. If you would like to filter out certain WAC/Static bands, you can filter them using the BAND_FILTER variable in the config. **See the README in the [LFM repo](https://github.com/nasa-nccs-hpda/lfm) for more info.** 

## Getting started

### Downloading
You can get started with this notebook by downloading it with:

```bash
wget https://raw.githubusercontent.com/nasa-nccs-hpda/lfm/refs/heads/main/notebooks/cube_inference_sseg.ipynb
```

### Code dependencies -- terminal command
The notebook requires some code from the [LFM repo](https://github.com/nasa-nccs-hpda/lfm/tree/main). To install this code, navigate to the top of the JupyterHub user interface, and click on the box with the + symbol. Then scroll down, and choose the "Terminal" (under Other section, first option). 

Then, if it's your first time using any of the LFM notebooks, run:

```bash
cd path/to/lfm/notebooks
git clone https://github.com/nasa-nccs-hpda/lfm.git
```

Or, if you've already run the above command, run:

```bash
cd path/to/lfm/notebooks
git pull https://github.com/nasa-nccs-hpda/lfm/tree/main
```

This will get you the most up-to-date code to support the notebook. 

### Python installs
You will notice 2 folders below that install dependencies for you; one removes your "local python installs", and the other installs things locally. **This is completely harmless, and don't worry about anything important getting deleted**. Local installs are very easy to get back using pip or conda, we are just deleting previous installs here to make sure we have a clean environment to work with. 

**See the README in the [repo](https://github.com/nasa-nccs-hpda/lfm)** for more info on how to use this notebook, and more on the process of training the model. 

## Imports, Dino Repo Clone

In [1]:
!rm -rf ~/.local/lib/python*

In [2]:
!pip install rasterio transformers torchmetrics # For H100 environment

In [3]:
import os
import sys
import torch
import subprocess
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

%matplotlib inline

/usr/local/lib/python3.12/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [4]:
repo_dir = "lfm"
sys.path.insert(0, os.path.join(os.getcwd(), repo_dir))

from lfm.tasks.sem_segmentation.sseg_model import DINOSegmentation, load_dinov3_encoder
from lfm.tasks.sem_segmentation.sseg_dataset import get_dataloaders
from lfm.tasks.sem_segmentation.sseg_driver import train_model

Installing termcolor (required by DINOv3)...
✓ Termcolor installed successfully


## Main workflow

1. Define user-configured variables
2. Create dataloaders from files on /explore/nobackup/.
3. Load DinoV3 encoder, create encoder/decoder finetuning model.
4. Train model, print training stats, and visualize results. 

### User Config

#### Paths
`INPUT_ROOT_DIR`: there are multiple different dataset configurations under the lfm/model_inputs/300_300_inputs folder. Choose any of the band subdirectories in the 300_300_inputs folder, and leave the other variables under "data paths" (IMAGE_DIR, LABEL_DIR, etc) as they are. 

`OUTPUT_DIR`: this is a relative path, so by default the outputs (visualizations, model checkpoints, dataset statistics) will go to the same folder as the notebook. 

#### Dataset parameters
`MAX_SAMPLES`: number of training samples to look for in INPUT_ROOT_DIR. 

`TRAIN_SPLIT`: weight of how many training samples versus validation samples; the default is 0.8, or 80% training.

#### Training hyperparameters
`BATCH_SIZE`: best to leave this at 16 to conserve VRAM, especially at higher number of input bands (>7). 

`NUM_EPOCHS`: default is 100, the model tends to start to get its best results around here. Feel free to experiment with this.

`BASE_LR`,`WEIGHT_DECAY`: feel free to change these to adjust how aggressively the model tries to tune to each training batch. Higher means more aggressive model learning, but can also mean the model has a harder time converging to the correct result. 

#### Model hyperparameters
`FREEZE_ENCODER`: whether to keep the Dino backbone frozen; we found better results with False, but feel free to try with freezing set to true. 

`NUM_BANDS`: number of bands to include in the input. Currently supported are 3/5/7/12-band inputs.

`BAND_FILTER`: list of band indices (0-indexed) to use in the dataset; for example, if I want to use only VIS bands, I would supply [0, 1, 2, 3, 4]. Band ordering is the following for the 12-band dataset: VIS (0-4), UV (5, 6), KAGUYA STATIC (7-11). 

In [5]:
# Local image of DinoV3 encoder
weights_local_checkpoint = '/explore/nobackup/projects/lfm/model_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'

# Data paths
INPUT_ROOT_DIR = "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/sem_seg"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs/sem_seg"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Dataset parameters
MAX_SAMPLES = 500  # Set to None to use all samples, or an integer to limit
TRAIN_SPLIT = 0.8  # 80% train, 20% validation

# Training hyperparameters
BATCH_SIZE = 16  # Number of images fed into the model at a time
NUM_EPOCHS = 100  # 10 is the default, increase for more model learning
BASE_LR = 5e-5  # Starting learning rate
WEIGHT_DECAY = 1e-3  # Weight decay of optimizer

# Model parameters
FREEZE_ENCODER = False  # Whether to keep the DinoV3 weights frozen
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  # Bands to keep, in order of filtering

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Output directory: ./outputs/sem_seg
Using device: cuda


### Create dataloaders

In [6]:
# ============================================================================
# CREATE DATALOADERS
# ============================================================================

print("\n" + "="*60)
print("STEP 1: Creating dataloaders.")
print("="*60)

train_loader, val_loader, weight_assignments = get_dataloaders(
    base_dir=INPUT_ROOT_DIR,
    batch_size=BATCH_SIZE,
    train_split=TRAIN_SPLIT,
    max_samples=MAX_SAMPLES,
    seed=42,
    stats_save_dir=OUTPUT_DIR,
    debug=True,
    band_filter=BAND_FILTER,
    output_dir=OUTPUT_DIR,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")


STEP 1: Creating dataloaders.
Filtered inputs and mean/std to channels: [0, 2, 1, 3]
Limited to 500 samples
Found 500 matched image-label pairs
Dataset configured for 4 channel(s)
Saved 100 validation paths to ./outputs/sem_seg/val_paths.txt
Train samples: 400
Val samples: 100
Train batches: 25
Val batches: 7


### Load Encoder and Create Model

In [7]:
print("\n" + "="*60)
print("STEP 2: Loading DINO encoder and creating model.")
print("="*60)

encoder = load_dinov3_encoder(weights_local_checkpoint, device)


STEP 2: Loading DINO encoder and creating model.
Loading model from /explore/nobackup/projects/lfm/model_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth


Using cache found in /home/ajkerr1/.cache/torch/hub/facebookresearch_dinov3_main
/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Encoder loaded with pretrained weights.


In [8]:
# Create model with DINO segmentation head, UNet decoder (see model.py)
print("Creating DINO segmentation model with UNet decoder...")
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    freeze_encoder=FREEZE_ENCODER,
    weight_assignments=weight_assignments
).to(device)

Creating DINO segmentation model with UNet decoder...
Modifying input weights for 4 bands...
Applied flexible embedding approach: ['blue', '0.7*red+0.3*green', 'green', 'red']
Encoder unfrozen! Full model will be trained.


### Run Training

In [9]:
print("\n" + "="*60)
print("Starting training.")
print("="*60)

train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    device=device,
)


Starting training.
Using device: cuda
Using loss function: focal_dice
Loss function: FocalDiceLoss

Starting training for 100 epochs...
Starting from epoch: 1
Checkpoints will be saved every 10 epochs to: ./outputs/sem_seg/checkpoints
Visualizations will be saved every 10 epochs to: ./outputs/sem_seg/visualizations

MODEL PARAMETER SUMMARY
Encoder:
  Trainable: 303,418,368 / 303,418,368 (100.00%)

Decoder:
  Trainable: 5,924,610 / 5,924,610 (100.00%)

Combined Model:
  Trainable: 309,342,978 / 309,342,978 (100.00%)


Epoch 1/100


Epoch 1/100 [Train]:   0%|          | 0/25 [00:00<?, ?it/s]/explore/nobackup/people/ajkerr1/Lunar_FM/latest_repo/lfm/lfm/tasks/sem_segmentation/sseg_driver.py:402: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/native/Scalar.cpp:22.)
  total_loss += loss.item()
Epoch 1/100 [Val]: 100%|██████████| 7/7 [00:05<00:00,  1.24it/s, loss=0.524]



Epoch 1 Summary:
  Train Loss: 0.5705
  Val Loss:   0.5497
  LR:         0.000010
  Time:       26.38s
Saved checkpoint to: ./outputs/sem_seg/checkpoints/best_model.pt
  Saved best model (val_loss: 0.5497)

Epoch 2/100


Epoch 2/100 [Val]: 100%|██████████| 7/7 [00:00<00:00,  8.26it/s, loss=0.491]



Epoch 2 Summary:
  Train Loss: 0.5599
  Val Loss:   0.5284
  LR:         0.000014
  Time:       4.93s


KeyboardInterrupt: 

## Display some of the output visualizations

The training of the model is already producing some visualizations every N epochs.
Here we open some of the visualizations to look at them from the notebook.

In [ ]:
visualization_dir = os.path.join(OUTPUT_DIR, 'visualizations')
visualization_filenames = sorted(glob(os.path.join(visualization_dir, '*.png')))

In [ ]:
for vis_filename in visualization_filenames:
    img = mpimg.imread(vis_filename)
    plt.figure(figsize=(16, 14))
    plt.imshow(img)
    plt.axis("off")
    plt.show()